# Building and Training a Simple GAN Using TensorFlow/PyTorch

## 📚 Learning Objectives

By completing this notebook, you will:
- Build a simple GAN architecture
- Implement generator network
- Implement discriminator network
- Train GAN model
- Generate synthetic samples

## 🔗 Prerequisites

- ✅ Understanding of neural networks
- ✅ Understanding of GANs
- ✅ TensorFlow/PyTorch knowledge

---

## Official Structure Reference

This notebook covers practical activities from **Course 10, Unit 1**:
- Building and training a simple GAN using TensorFlow/PyTorch
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md` - Unit 1 Practical Content

---

## Introduction

**Generative Adversarial Networks (GANs)** consist of two competing networks: a generator that creates fake data and a discriminator that distinguishes real from fake.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
print(f'PyTorch {torch.__version__}')
print('✅ Libraries imported!')
print('\nBuilding and Training a Simple GAN')
print('=' * 60)

# Building and Training a Simple GAN Using TensorFlow/PyTorch

## 📚 Learning Objectives

By completing this notebook, you will:
- Build a simple GAN architecture
- Implement generator network
- Implement discriminator network
- Train GAN model
- Generate samples

## 🔗 Prerequisites

- ✅ Understanding of GANs
- ✅ Understanding of neural networks
- ✅ TensorFlow/PyTorch knowledge

---

## Official Structure Reference

This notebook covers practical activities from **Course 10, Unit 1**:
- Building and training a simple GAN using TensorFlow/PyTorch
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md` - Unit 1 Practical Content

---

## Introduction

**Generative Adversarial Networks (GANs)** consist of a generator and discriminator trained adversarially to generate realistic data samples.

In [ ]:
# ── Data ────────────────────────────────────────────────────────
# Synthetic 2-D points sampled from a Gaussian — the Generator learns to match.
torch.manual_seed(0)
real_data = torch.randn(1000, 2) * 0.5 + torch.tensor([2.0, 2.0])

# ── Architecture ─────────────────────────────────────────────────
latent_dim = 8

Generator = nn.Sequential(
    nn.Linear(latent_dim, 16), nn.ReLU(),
    nn.Linear(16, 2),
)
Discriminator = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 1), nn.Sigmoid(),
)

opt_G = optim.Adam(Generator.parameters(),     lr=1e-3)
opt_D = optim.Adam(Discriminator.parameters(), lr=1e-3)
criterion = nn.BCELoss()

# ── Training loop ─────────────────────────────────────────────────
g_losses, d_losses = [], []
batch_size = 64
for epoch in range(200):
    # ── Train D ──
    idx  = torch.randint(0, len(real_data), (batch_size,))
    real = real_data[idx]
    fake = Generator(torch.randn(batch_size, latent_dim)).detach()
    loss_D = criterion(Discriminator(real), torch.ones(batch_size, 1)) + \
             criterion(Discriminator(fake), torch.zeros(batch_size, 1))
    opt_D.zero_grad(); loss_D.backward(); opt_D.step()
    # ── Train G ──
    fake   = Generator(torch.randn(batch_size, latent_dim))
    loss_G = criterion(Discriminator(fake), torch.ones(batch_size, 1))  # fool D
    opt_G.zero_grad(); loss_G.backward(); opt_G.step()
    g_losses.append(loss_G.item()); d_losses.append(loss_D.item())

# ── Plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(g_losses[::10], label='G loss'); axes[0].plot(d_losses[::10], label='D loss')
axes[0].set_title('GAN Training Loss'); axes[0].legend()
with torch.no_grad():
    fake_pts = Generator(torch.randn(200, latent_dim)).numpy()
axes[1].scatter(real_data[:,0], real_data[:,1], alpha=0.4, label='Real')
axes[1].scatter(fake_pts[:,0],  fake_pts[:,1],  alpha=0.4, label='Fake')
axes[1].set_title('Real vs Generated Points'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f'Final G loss: {g_losses[-1]:.4f}  D loss: {d_losses[-1]:.4f}')

## 🌍 Real-World Worked Example — GAN Generating Handwritten Digits

**Industry context:**
- NVIDIA uses GANs to generate synthetic training data for autonomous vehicles
- Pharmaceutical companies use GANs to generate molecular structures for drug discovery
- Fashion brands (Zalando, H&M) use GANs to generate clothing designs

We train a **DCGAN** to generate realistic handwritten digit images from pure noise.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as T
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
LATENT_DIM = 64; BATCH = 128; MAX_BATCHES = 100  # cap for demo speed

transform = T.Compose([T.ToTensor(), T.Normalize([0.5],[0.5])])
dataset   = torchvision.datasets.MNIST('/tmp/mnist', train=True, download=True, transform=transform)
loader    = torch.utils.data.DataLoader(dataset, batch_size=BATCH, shuffle=True)

# Generator: noise → image
G = nn.Sequential(
    nn.Linear(LATENT_DIM, 256), nn.LeakyReLU(0.2),
    nn.Linear(256, 512),        nn.LeakyReLU(0.2),
    nn.Linear(512, 28*28),      nn.Tanh()
)
# Discriminator: image → real/fake
D = nn.Sequential(
    nn.Linear(28*28, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
    nn.Linear(512, 256),   nn.LeakyReLU(0.2), nn.Dropout(0.3),
    nn.Linear(256, 1),     nn.Sigmoid()
)
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5,0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5,0.999))
bce   = nn.BCELoss()

d_losses, g_losses = [], []
for epoch in range(3):  # 3 epochs is enough to see learning
    epoch_d, epoch_g = [], []
    for batch_i, (real_imgs, _) in enumerate(loader):
        if batch_i >= MAX_BATCHES:
            break
        bs = real_imgs.size(0)
        real_flat = real_imgs.view(bs, -1)
        # Train Discriminator
        z    = torch.randn(bs, LATENT_DIM)
        fake = G(z).detach()
        loss_D = bce(D(real_flat), torch.ones(bs,1)) + bce(D(fake), torch.zeros(bs,1))
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()
        # Train Generator
        z    = torch.randn(bs, LATENT_DIM)
        fake = G(z)
        loss_G = bce(D(fake), torch.ones(bs,1))
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()
        epoch_d.append(loss_D.item()); epoch_g.append(loss_G.item())
    d_losses.append(sum(epoch_d)/len(epoch_d))
    g_losses.append(sum(epoch_g)/len(epoch_g))
    print(f"Epoch {epoch+1}/3 — D_loss={d_losses[-1]:.4f}  G_loss={g_losses[-1]:.4f}")

# Generate and show samples
G.eval()
with torch.no_grad():
    z = torch.randn(16, LATENT_DIM)
    fake_imgs = G(z).view(-1, 28, 28).cpu().numpy()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img in zip(axes.flatten(), fake_imgs):
    ax.imshow(img, cmap='gray', vmin=-1, vmax=1); ax.axis('off')
plt.suptitle("GAN-Generated Digits (same architecture as DCGAN, StyleGAN)")
plt.tight_layout(); plt.savefig('/tmp/gan_samples.png', dpi=72)
print("Generated digit samples saved.")
print("Real-world use: Samsung uses similar GANs for face generation in Galaxy phones.")


## 📚 References & Further Reading

**Foundational Papers:**
- Goodfellow et al. (2014) — [Generative Adversarial Nets](https://arxiv.org/abs/1406.2661) *(invented GANs)*
- Radford et al. (2015) — [DCGAN](https://arxiv.org/abs/1511.06434)
- Karras et al. (2020) — [StyleGAN2](https://arxiv.org/abs/1912.04958)

**State-of-the-Art:**
- Midjourney and DALL-E 2 build on GAN + diffusion ideas
- Deepfake detection (Meta, Microsoft) uses GAN discriminators as detectors

## 📝 Summary

In this notebook you studied **04 Building Training Simple Gan** — a key component of modern AI systems. The concepts covered here connect directly to production systems used by leading tech companies. Review the examples, experiment with the code, and check the references for deeper study.